[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/main/workshop/04_linking_and_ambiguity.ipynb)

# 04 · Linking, ambiguity and historical geography

**Spatial Humanities 2026 workshop**

**Official workshop title:** *AI and NLP for Spatial Humanities: From Manual Annotation to LLM-Assisted Interpretation*

This notebook moves from **recognition** ("this string looks like a place") to **resolution** ("which place does it refer to?").

## Learning goals
By the end, you should be able to:
- distinguish named-entity recognition from entity linking/geocoding;
- inspect candidate ambiguity instead of hiding it;
- understand why a coordinate is an interpretive decision;
- identify cases where present-day gazetteers are historically inappropriate;
- preserve source forms and route uncertain resolutions to human review.

> **Key message:** A coordinate is an interpretation, not simply an annotation.

In [1]:
!wget -qO sh2026_setup.py https://raw.githubusercontent.com/IgnatiusEzeani/spatial-humanities-2026/main/workshop/sh2026_setup.py

import sh2026_setup as sh
ctx = sh.setup()

# Bound from the shared context: the cells below were written against these.
repo_dir = ctx.repo
data_dir = ctx.data

import json
REPO = "https://github.com/IgnatiusEzeani/spatio-textual.git"


Reusing existing checkout at /home/ezeani/workspace/spatial-humanities-2026
Dependencies already installed in this runtime.

Ready in 0s.
  repo    : /home/ezeani/workspace/spatial-humanities-2026
  commit  : 798f2be
  data    : /home/ezeani/workspace/spatial-humanities-2026/workshop/data
  outputs : /home/ezeani/workspace/spatial-humanities-2026/sh2026_outputs
  route   : CPU only, no API key needed

If this cell failed, put your hand up. Do not re-run it more than once.


## 1. Recognition is not resolution

Suppose an NER model marks **Cambridge** as a place. That is only the first question.

Resolution asks:
- Cambridge, England?
- Cambridge, Massachusetts?
- another Cambridge?

A gazetteer typically returns one or more candidate records. Choosing one candidate may require document context, historical knowledge, or a human decision.

In [2]:
from spatio_textual.geocode import GeoResolver

resolver = GeoResolver(max_candidates=5)

examples = ["London", "Cambridge", "Amsterdam", "Czechoslovakia", "Atlantis"]
rows = []
for name in examples:
    result = resolver.resolve(name, label="GPE", context=f"The text mentions {name}.")
    rows.append({"source_text": name, **(result or {})})

rows

[{'source_text': 'London',
  'resolved_name': 'London',
  'lat': 51.50853,
  'lon': -0.12574,
  'place_type_resolved': 'CITY',
  'resolution_status': 'resolved_ambiguous',
  'geo_source': 'geonamescache:city',
  'geo_confidence': 0.62,
  'ambiguous': True,
  'candidates_count': 2,
  'candidates': [{'name': 'London',
    'countrycode': 'GB',
    'population': 8961989,
    'lat': 51.50853,
    'lon': -0.12574,
    'geonameid': 2643743},
   {'name': 'London',
    'countrycode': 'CA',
    'population': 422324,
    'lat': 42.98339,
    'lon': -81.23304,
    'geonameid': 6058560}],
  'geonameid': 2643743,
  'countrycode': 'GB'},
 {'source_text': 'Cambridge',
  'resolved_name': 'Cambridge',
  'lat': 52.2,
  'lon': 0.11667,
  'place_type_resolved': 'CITY',
  'resolution_status': 'resolved_ambiguous',
  'geo_source': 'geonamescache:city',
  'geo_confidence': 0.62,
  'ambiguous': True,
  'candidates_count': 4,
  'candidates': [{'name': 'Cambridge',
    'countrycode': 'GB',
    'population': 1456

**Inspect the output carefully.** Useful audit fields include:

- `resolved_name`
- `resolution_status`
- `geo_source`
- `geo_confidence`
- `ambiguous`
- `candidates_count`
- `candidates`

The important question is not merely *"did we get coordinates?"* but *"what evidence justified these coordinates?"*

In [3]:
# A compact table for inspection.
import pandas as pd

cols = ["source_text", "resolved_name", "resolution_status", "place_type_resolved", "lat", "lon", "geo_confidence", "ambiguous", "candidates_count"]
df = pd.DataFrame(rows)
for col in cols:
    if col not in df.columns:
        df[col] = None
display(df[cols].fillna(""))

,source_text,resolved_name,resolution_status,place_type_resolved,lat,lon,geo_confidence,ambiguous,candidates_count
0,London,London,resolved_ambiguous,CITY,51.50853,-0.12574,0.62,True,2
1,Cambridge,Cambridge,resolved_ambiguous,CITY,52.2,0.11667,0.62,True,4
2,Amsterdam,Amsterdam,resolved_ambiguous,CITY,52.37403,4.88969,0.62,True,2
3,Czechoslovakia,Czechoslovakia,unresolved,HISTORICAL_POLITY,,,0.00,True,0
4,Atlantis,Atlantis,resolved,CITY,-33.56668,18.48335,0.88,False,1


## 2. Ambiguity as a first-class result

If a name has several plausible candidates, `GeoResolver` marks the result as ambiguous rather than pretending that ranking equals certainty.

Try changing the preferred country below. `prefer_country` is a **ranking preference**, not evidence that the preferred candidate is historically correct.

In [4]:
for preference in [None, "GB", "US"]:
    r = GeoResolver(prefer_country=preference, max_candidates=5)
    out = r.resolve("Cambridge", label="GPE", context="I travelled from Cambridge to London.")
    print("\nPreference:", preference)
    print(json.dumps(out, indent=2, ensure_ascii=False))


Preference: None
{
  "resolved_name": "Cambridge",
  "lat": 52.2,
  "lon": 0.11667,
  "place_type_resolved": "CITY",
  "resolution_status": "resolved_ambiguous",
  "geo_source": "geonamescache:city",
  "geo_confidence": 0.62,
  "ambiguous": true,
  "candidates_count": 4,
  "candidates": [
    {
      "name": "Cambridge",
      "countrycode": "GB",
      "population": 145674,
      "lat": 52.2,
      "lon": 0.11667,
      "geonameid": 2653941
    },
    {
      "name": "Cambridge",
      "countrycode": "CA",
      "population": 129920,
      "lat": 43.3601,
      "lon": -80.31269,
      "geonameid": 5913695
    },
    {
      "name": "Cambridge",
      "countrycode": "US",
      "population": 110402,
      "lat": 42.3751,
      "lon": -71.10561,
      "geonameid": 4931972
    },
    {
      "name": "Cambridge",
      "countrycode": "NZ",
      "population": 15192,
      "lat": -37.87822,
      "lon": 175.4402,
      "geonameid": 6240770
    }
  ],
  "geonameid": 2653941,
  "countrycode


Preference: US
{
  "resolved_name": "Cambridge",
  "lat": 52.2,
  "lon": 0.11667,
  "place_type_resolved": "CITY",
  "resolution_status": "resolved_ambiguous",
  "geo_source": "geonamescache:city",
  "geo_confidence": 0.62,
  "ambiguous": true,
  "candidates_count": 4,
  "candidates": [
    {
      "name": "Cambridge",
      "countrycode": "GB",
      "population": 145674,
      "lat": 52.2,
      "lon": 0.11667,
      "geonameid": 2653941
    },
    {
      "name": "Cambridge",
      "countrycode": "CA",
      "population": 129920,
      "lat": 43.3601,
      "lon": -80.31269,
      "geonameid": 5913695
    },
    {
      "name": "Cambridge",
      "countrycode": "US",
      "population": 110402,
      "lat": 42.3751,
      "lon": -71.10561,
      "geonameid": 4931972
    },
    {
      "name": "Cambridge",
      "countrycode": "NZ",
      "population": 15192,
      "lat": -37.87822,
      "lon": 175.4402,
      "geonameid": 6240770
    }
  ],
  "geonameid": 2653941,
  "countrycode":

### Discussion

1. Should population be the default ranking heuristic?
2. What evidence in the surrounding text could disambiguate a place?
3. When should the system refuse to decide?
4. Should a human correction overwrite the machine suggestion or be stored alongside it?

For this project, the preferred design is **append-only provenance**: keep the model suggestion and record the human decision separately.

## 3. Historical geography: why "unresolved" can be the better answer

Present-day gazetteers are not historical GIS systems. A historical polity should not be silently collapsed onto a current state simply because a modern database requires one coordinate.

The SH2026 branch therefore preserves **Czechoslovakia** as the source string and marks it as a historical polity requiring time-aware or human resolution.

In [5]:
historical = resolver.resolve(
    "Czechoslovakia",
    label="GPE",
    context="The narrator described a journey through Czechoslovakia before later border changes."
)
print(json.dumps(historical, indent=2, ensure_ascii=False))

assert historical["resolved_name"] == "Czechoslovakia"
assert historical["place_type_resolved"] == "HISTORICAL_POLITY"
assert historical["resolution_status"] == "unresolved"
assert historical["lat"] is None and historical["lon"] is None

{
  "resolved_name": "Czechoslovakia",
  "lat": null,
  "lon": null,
  "place_type_resolved": "HISTORICAL_POLITY",
  "resolution_status": "unresolved",
  "geo_source": "historical_name:preserved",
  "geo_confidence": 0.0,
  "ambiguous": true,
  "candidates_count": 0,
  "candidates": [],
  "historical_name": true,
  "review_reason": "Historical polity requires time-aware/human resolution; source form preserved."
}


This gives us an important methodological rule:

> **Preserve the historical source form before attempting modern normalization.**

A time-aware gazetteer could later attach period-specific geometries, but that is a different task from ordinary present-day geocoding.

## 4. A small human-review record

Rather than replacing the machine output, create a review record that preserves:
- source string;
- machine suggestion;
- reviewer decision;
- reason;
- timestamp/version if available.

In [6]:
from datetime import datetime, timezone

machine = resolver.resolve("Cambridge", label="GPE", context="I left Cambridge and travelled to London.")

review = {
    "source_text": "Cambridge",
    "context": "I left Cambridge and travelled to London.",
    "machine_resolution": machine,
    "human_decision": {
        "status": "accepted_for_teaching_example",
        "resolved_name": machine.get("resolved_name") if machine else None,
        "reason": "Illustrative review only; real research requires document-level contextual evidence.",
        "reviewed_at": datetime.now(timezone.utc).isoformat(),
    }
}

print(json.dumps(review, indent=2, ensure_ascii=False))

{
  "source_text": "Cambridge",
  "context": "I left Cambridge and travelled to London.",
  "machine_resolution": {
    "resolved_name": "Cambridge",
    "lat": 52.2,
    "lon": 0.11667,
    "place_type_resolved": "CITY",
    "resolution_status": "resolved_ambiguous",
    "geo_source": "geonamescache:city",
    "geo_confidence": 0.62,
    "ambiguous": true,
    "candidates_count": 4,
    "candidates": [
      {
        "name": "Cambridge",
        "countrycode": "GB",
        "population": 145674,
        "lat": 52.2,
        "lon": 0.11667,
        "geonameid": 2653941
      },
      {
        "name": "Cambridge",
        "countrycode": "CA",
        "population": 129920,
        "lat": 43.3601,
        "lon": -80.31269,
        "geonameid": 5913695
      },
      {
        "name": "Cambridge",
        "countrycode": "US",
        "population": 110402,
        "lat": 42.3751,
        "lon": -71.10561,
        "geonameid": 4931972
      },
      {
        "name": "Cambridge",
        "

## 5. Resolution confidence is not historical truth

A numerical confidence score can summarize the resolver's own evidence, but it does **not** mean:
- 95% probability that a historical interpretation is correct;
- 95% agreement among scholars;
- 95% certainty that the narrator intended that place.

Confidence is meaningful only when its provenance and calculation are understood.

## 6. Exercise: decide what should be mapped

For each phrase, decide whether you would:

1. map directly;
2. map with an ambiguity flag;
3. preserve without coordinates;
4. request human/historical review.

- "London"
- "Cambridge"
- "Czechoslovakia"
- "the village"
- "beyond the river"
- "home"

Notice that several spatially meaningful expressions are **not failures** simply because they cannot be represented as one point on a modern basemap.

In [7]:
decision_template = [
    {"expression": "London", "decision": "", "reason": ""},
    {"expression": "Cambridge", "decision": "", "reason": ""},
    {"expression": "Czechoslovakia", "decision": "", "reason": ""},
    {"expression": "the village", "decision": "", "reason": ""},
    {"expression": "beyond the river", "decision": "", "reason": ""},
    {"expression": "home", "decision": "", "reason": ""},
]
pd.DataFrame(decision_template)

,expression,decision,reason
0,London,,
1,Cambridge,,
2,Czechoslovakia,,
3,the village,,
4,beyond the river,,
5,home,,


## 7. Take-away

We now have four distinct layers:

**Recognition → Resolution → Relation → Interpretation**

A system may perform very well at recognition while still being uncertain at resolution. Historical and experiential spatial references can also be meaningful without a defensible coordinate.

**Next:** affect and narrator-centred events, where the same audit principle becomes even more important: computational labels are signals for analysis, not psychological ground truth.